# Assignment 3 — Data Exploration
Explore the lyrics CSV, MIDI files, and Word2Vec embeddings before building the model.

In [1]:
import pandas as pd
import pretty_midi
from gensim.models import KeyedVectors
import torch
import os

print(f"PyTorch version: {torch.__version__}")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.12.0
Using device: mps


## 1. Lyrics CSV

In [3]:
# No header row; train set has trailing empty columns — keep only the 3 real ones
cols = ['artist', 'song', 'lyrics']

df_train = pd.read_csv("../data/lyrics_train_set.csv", header=None, usecols=[0, 1, 2], names=cols)
df_test  = pd.read_csv("../data/lyrics_test_set.csv",  header=None, usecols=[0, 1, 2], names=cols)

print(f"Train shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")
display(df_train.head())
display(df_test.head())

# Show how '&' separates lyric lines for the first training song
sample = df_train.iloc[0]
lines = [l.strip() for l in sample['lyrics'].split('&') if l.strip()]
print(f"\nFirst song: '{sample['song']}' by {sample['artist']}")
print(f"Number of lyric lines: {len(lines)}")
print("First 5 lines:")
for line in lines[:5]:
    print(f"  {line}")

Train shape: (600, 3)
Test  shape: (5, 3)


,artist,song,lyrics
0,elton john,candle in the wind,goodbye norma jean & though i never knew you a...
1,gerry rafferty,baker street,winding your way down on baker street & lite i...
2,gerry rafferty,right down the line,you know i need your love & you've got that ho...
3,2 unlimited,tribal dance,come on check it out ya'll & (come on come on!...
4,2 unlimited,let the beat control your body,let the beat control your body & let the beat ...


,artist,song,lyrics
0,the bangles,eternal flame,close your eyes give me your hand darling & d...
1,billy joel,honesty,if you search for tenderness & it isn't hard ...
2,cardigans,lovefool,dear i fear we're facing a problem & you love...
3,aqua,barbie girl,hiya barbie & hi ken! & do you want to go for...
4,blink 182,all the small things,all the small things & true care truth brings...



First song: 'candle in the wind' by elton john
Number of lyric lines: 50
First 5 lines:
  goodbye norma jean
  though i never knew you at all
  you had the grace to hold yourself
  while those around you crawled
  they crawled out of the woodwork


## 2. MIDI File Inspection

In [4]:
# MIDI files are named: Artist_Name_-_Song_Title.mid
midi_dir = "../data/midi_files"
midi_files = sorted(os.listdir(midi_dir))
print(f"Total MIDI files: {len(midi_files)}")
print(f"Sample names: {midi_files[:5]}")

# Load the first MIDI file
midi_path = os.path.join(midi_dir, midi_files[0])
midi = pretty_midi.PrettyMIDI(midi_path)
print(f"\nLoaded: {midi_files[0]}")
print(f"Duration: {midi.get_end_time():.2f} seconds")

# Instruments
print(f"\nInstruments ({len(midi.instruments)}):")
for inst in midi.instruments:
    print(f"  [{inst.program}] {inst.name or 'Unnamed'} | "
          f"drum={inst.is_drum} | notes={len(inst.notes)}")

# Key signatures
print(f"\nKey signature changes: {midi.key_signature_changes}")

# Time signatures
print(f"Time signature changes: {midi.time_signature_changes}")

# First 10 notes of the first instrument
if midi.instruments:
    inst = midi.instruments[0]
    print(f"\nFirst 10 notes of '{inst.name or 'Instrument 0'}':")
    for note in inst.notes[:10]:
        print(f"  pitch={note.pitch:3d} | start={note.start:.3f}s | "
              f"end={note.end:.3f}s | velocity={note.velocity}")

Total MIDI files: 625
Sample names: ['1910_Fruitgum_Company_-_Simon_Says.mid', '2_Unlimited_-_Get_Ready_for_This.mid', '2_Unlimited_-_Let_the_Beat_Control_Your_Body.mid', '2_Unlimited_-_Tribal_Dance.mid', '2_Unlimited_-_Twilight_Zone.mid']

Loaded: 1910_Fruitgum_Company_-_Simon_Says.mid
Duration: 158.44 seconds

Instruments (11):
  [18] Organ | drum=False | notes=1894
  [34] Bass | drum=False | notes=499
  [48] Strings | drum=False | notes=155
  [67] Melody | drum=False | notes=268
  [25] 6 String | drum=False | notes=751
  [0] 12 String | drum=False | notes=670
  [0] Chips | drum=False | notes=456
  [61] Brass | drum=False | notes=390
  [53] Syn Vox | drum=False | notes=372
  [0] Kick | drum=True | notes=1145
  [16] Solo Organ | drum=False | notes=78

Key signature changes: [KeySignature(key_number=0, time=0.0)]
Time signature changes: [TimeSignature(numerator=4, denominator=4, time=0.0)]

First 10 notes of 'Organ':
  pitch= 61 | start=4.000s | end=4.343s | velocity=127
  pitch= 73 | 

## 3. Word2Vec Embeddings

In [6]:
# Download Google News Word2Vec from:
# saf
# Place the binary at: ../data/GoogleNews-vectors-negative300.bin
w2v_path = "../data/GoogleNews-vectors-negative300.bin"

if not os.path.exists(w2v_path):
    print("Word2Vec binary not found at:", w2v_path)
    print("Download from Google Drive (link above) and place it in ../data/")
else:
    print("Loading Word2Vec (this takes ~1 min)...")
    wv = KeyedVectors.load_word2vec_format(w2v_path, binary=True)
    print(f"Vocabulary size: {len(wv):,}")
    print(f"Vector dimensions: {wv.vector_size}")

    test_word = "love"
    vec = wv[test_word]
    print(f"\nVector for '{test_word}': shape={vec.shape}, dtype={vec.dtype}")
    print(f"First 10 values: {vec[:10]}")

    print(f"\nTop-5 most similar to '{test_word}':")
    for word, score in wv.most_similar(test_word, topn=5):
        print(f"  {word:<20} {score:.4f}")

Loading Word2Vec (this takes ~1 min)...
Vocabulary size: 3,000,000
Vector dimensions: 300

Vector for 'love': shape=(300,), dtype=float32
First 10 values: [ 0.10302734 -0.15234375  0.02587891  0.16503906 -0.16503906  0.06689453
  0.29296875 -0.26367188 -0.140625    0.20117188]

Top-5 most similar to 'love':
  loved                0.6908
  adore                0.6817
  loves                0.6619
  passion              0.6101
  hate                 0.6004
